# **Import** **Libraries**


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


In [4]:
raw_data=pd.read_csv("/content/sample_data/swiftchain_logistics_450k.csv")

Basic Information of dataset

In [ ]:
raw_data.shape

(450800, 28)

**Phase1 ---EDA Analysis---**

In [5]:
print("info:",raw_data.info())
print("describe:",raw_data.describe())




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450800 entries, 0 to 450799
Data columns (total 28 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Shipment_ID             450800 non-null  object 
 1   Order_Date              450800 non-null  object 
 2   Promised_Delivery_Date  450800 non-null  object 
 3   Actual_Delivery_Date    437103 non-null  object 
 4   Delay_Days              437103 non-null  float64
 5   Delay_Status            437103 non-null  object 
 6   On_Time_Flag            437103 non-null  float64
 7   Origin_City             450800 non-null  object 
 8   Origin_State            450800 non-null  object 
 9   Origin_Region           450800 non-null  object 
 10  Destination_City        450800 non-null  object 
 11  Destination_State       450800 non-null  object 
 12  Destination_Region      450800 non-null  object 
 13  Route_ID                450800 non-null  object 
 14  Distance_Miles      

By seeing above info we can see clearly **nulls in dataset**(Actual_Delivery_Date,Delay_Days,Delay_Status ,On_Time_Flag,Weather_Condition,Traffic_Condition) &(Actual_Delivery_Date,Promised_Delivery_Date,Order_Date are **object type**)

In [10]:
raw_data.duplicated().sum()

np.int64(800)

From above we can see that we have **duplicates** in data

In [6]:
raw_data.columns

Index(['Shipment_ID', 'Order_Date', 'Promised_Delivery_Date',
       'Actual_Delivery_Date', 'Delay_Days', 'Delay_Status', 'On_Time_Flag',
       'Origin_City', 'Origin_State', 'Origin_Region', 'Destination_City',
       'Destination_State', 'Destination_Region', 'Route_ID', 'Distance_Miles',
       'Warehouse_ID', 'Warehouse_City', 'Warehouse_State', 'Carrier',
       'Ship_Mode', 'Product_Category', 'Product_Weight_kg', 'Order_Value',
       'Shipping_Cost', 'Customer_Segment', 'Customer_ID', 'Weather_Condition',
       'Traffic_Condition'],
      dtype='object')

So we have nulls and duplicates according to bussiness rule:nulls in (Actual_Delivery_Date,Delay_Days,Delay_Status ,On_Time_Flag) we are separating **into intransit.csv**. Weather_Condition
Traffic_Condition are filled by **"Unknown"**

In [11]:
in_transit=raw_data.loc[raw_data["Actual_Delivery_Date"].isnull()]
in_transit.to_csv("intransit.csv",index=False)
raw_data.drop(in_transit.index,inplace=True)
raw_data.drop_duplicates(inplace=True)

In [18]:
raw_data.isnull().sum().reset_index()


,index,0
0,Shipment_ID,0
1,Order_Date,0
2,Promised_Delivery_Date,0
3,Actual_Delivery_Date,0
4,Delay_Days,0
5,Delay_Status,0
6,On_Time_Flag,0
7,Origin_City,0
8,Origin_State,0
9,Origin_Region,0


Still we have nulls in Weather_Condition:21945
Traffic_Condition:17612 so we are filling with "Unknown"


In [27]:
raw_data['Weather_Condition']=raw_data['Weather_Condition'].fillna("Unknown")
raw_data['Traffic_Condition']=raw_data['Traffic_Condition'].fillna("Unknown")
print(f" nulls:{raw_data.isnull().sum()},dupicates:{raw_data.duplicated().sum()},shape:{raw_data.shape}")

 nulls:Shipment_ID               0
Order_Date                0
Promised_Delivery_Date    0
Actual_Delivery_Date      0
Delay_Days                0
Delay_Status              0
On_Time_Flag              0
Origin_City               0
Origin_State              0
Origin_Region             0
Destination_City          0
Destination_State         0
Destination_Region        0
Route_ID                  0
Distance_Miles            0
Warehouse_ID              0
Warehouse_City            0
Warehouse_State           0
Carrier                   0
Ship_Mode                 0
Product_Category          0
Product_Weight_kg         0
Order_Value               0
Shipping_Cost             0
Customer_Segment          0
Customer_ID               0
Weather_Condition         0
Traffic_Condition         0
dtype: int64,dupicates:0,shape:(436328, 28)


From above we have no nulls and dupilicates and dataset shape changed

In [23]:
for i in raw_data.columns:
  print(f"{i} :{raw_data[i].nunique()}")

Shipment_ID :436328
Order_Date :1461
Promised_Delivery_Date :1470
Actual_Delivery_Date :1519
Delay_Days :68
Delay_Status :5
On_Time_Flag :2
Origin_City :24
Origin_State :22
Origin_Region :4
Destination_City :24
Destination_State :22
Destination_Region :4
Route_ID :200
Distance_Miles :2951
Warehouse_ID :10
Warehouse_City :10
Warehouse_State :9
Carrier :5
Ship_Mode :4
Product_Category :8
Product_Weight_kg :49947
Order_Value :378584
Shipping_Cost :55321
Customer_Segment :3
Customer_ID :8000
Weather_Condition :6
Traffic_Condition :5


**Now Main analysis "Why delay is happening"**

In [33]:
Ontimepercent=raw_data.groupby(["On_Time_Flag"]).agg(
    total_orders=("On_Time_Flag","count")).reset_index()
Ontimepercent["deliverytime%"]=round(Ontimepercent['total_orders']/Ontimepercent['total_orders'].sum()*100,2)
print(Ontimepercent)



   On_Time_Flag  total_orders  deliverytime%
0           0.0        223098          51.13
1           1.0        213230          48.87


According to above info we on time delivery precentage is 48.87% and delay delivery percentage is 51.15%. So ontime is less than industry level 85% so we to find which causing delay

so i am starting from basic analayis( who what where and why),by seeing data i know some([Warehouse_ID,Product_Category,Carrier,Weather_Condition,Traffic_Condition,Ship_Mode,Distance_Miles])

In [53]:
cols=["Warehouse_ID","Product_Category","Carrier","Weather_Condition","Traffic_Condition","Ship_Mode","Route_ID","Origin_Region","Destination_Region"]
for i in cols:
  info= raw_data.groupby([i]).agg(
     Total_orders=("Shipment_ID","count"),
     ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
     delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())
    ).reset_index()
  info["ontime_per"]=round(info['ontime_orders']/info['Total_orders']*100,2)
  info["delayed_per"]=round(info['delayed_orders']/info['Total_orders']*100,2)
  print(info.sort_values(ascending=True,by="ontime_per"))





  Warehouse_ID  Total_orders  ontime_orders  delayed_orders  ontime_per  \
1       WH-002         43379          15071           28308       34.74   
7       WH-008         43449          15138           28311       34.84   
4       WH-005         43552          15300           28252       35.13   
3       WH-004         43184          23512           19672       54.45   
0       WH-001         43697          23837           19860       54.55   
5       WH-006         43654          23903           19751       54.76   
2       WH-003         43697          23955           19742       54.82   
9       WH-010         43946          24095           19851       54.83   
6       WH-007         43573          23973           19600       55.02   
8       WH-009         44197          24446           19751       55.31   

   delayed_per  
1        65.26  
7        65.16  
4        64.87  
3        45.55  
0        45.45  
5        45.24  
2        45.18  
9        45.17  
6        44.98  
8   

see in warehouse ontime% is 32 lowest(wh-002,wh-005,wh-008) the main issues warehouses.Carrier (DHL:37,Usps:46) so we have look into too,**main problem is Pharmaceuticals is getting delayed**. we also have route problem.

First we have find which warehouse and region warehouse causing problem:

In [60]:
warehouses=raw_data.groupby(["Warehouse_ID","Origin_Region"]).agg(
    total_orders=("Warehouse_ID","count"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())).reset_index()
warehouses["ontime_per"]=round(warehouses['ontime_orders']/warehouses['total_orders']*100,2)
warehouses["delayed_per"]=round(warehouses['delayed_orders']/warehouses['total_orders']*100,2)
warehouses.sort_values(ascending=True,by="ontime_per")

,Warehouse_ID,Origin_Region,total_orders,ontime_orders,delayed_orders,ontime_per,delayed_per
6,WH-002,South,9518,3246,6272,34.10,65.90
17,WH-005,East,12886,4453,8433,34.56,65.44
5,WH-002,East,13131,4542,8589,34.59,65.41
31,WH-008,West,12179,4218,7961,34.63,65.37
29,WH-008,East,12996,4512,8484,34.72,65.28
28,WH-008,Central,8662,3025,5637,34.92,65.08
4,WH-002,Central,8651,3028,5623,35.00,65.00
18,WH-005,South,9505,3332,6173,35.06,64.94
30,WH-008,South,9612,3383,6229,35.20,64.80
7,WH-002,West,12079,4255,7824,35.23,64.77


We found that warehouse is main reason it does not depend on region. now focus on carrier



In [65]:
carrier=raw_data.groupby(["Carrier"]).agg(
    total_orders=("Carrier","count"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())
).reset_index()
carrier["ontime_per"]=round(carrier['ontime_orders']/carrier['total_orders']*100,2)
carrier["delayed_per"]=round(carrier['delayed_orders']/carrier['total_orders']*100,2)

In [68]:
print(carrier.sort_values(ascending=True,by="ontime_per"))

  Carrier  total_orders  ontime_orders  delayed_orders  ontime_per  \
0     DHL         65444          24172           41272       36.94   
4    USPS         87714          40206           47508       45.84   
2  OnTrac         43706          21855           21851       50.00   
3     UPS        117887          60559           57328       51.37   
1   FedEx        121577          66438           55139       54.65   

   delayed_per  
0        63.06  
4        54.16  
2        50.00  
3        48.63  
1        45.35  


Base on above info DHL and usps is main problem, we imporve these two we can improve overall percent or assign orders to fedex is better.

Now combine both warehouses and carrier

In [82]:
warehouses_carriers=raw_data.groupby(["Warehouse_ID","Carrier"]).agg(
    total_orders=("Shipment_ID","count"),
    order_value=("Order_Value","mean"),
    shippingcost=("Shipping_Cost","mean"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())
).reset_index()
warehouses_carriers["ontime_per"]=round(warehouses_carriers['ontime_orders']/warehouses_carriers['total_orders']*100,2)
warehouses_carriers["delayed_per"]=round(warehouses_carriers['delayed_orders']/warehouses_carriers['total_orders']*100,2)

warehouses_carriers.sort_values(ascending=True,by="ontime_per")

def classify(ontime_per):
  if ontime_per<40:
    return "ClassC","Critical"

  elif ontime_per>=40 and ontime_per<=50:
    return "ClassB","Watch"

  else:
    return "ClassA","Excellent"
warehouses_carriers[["class","Performance"]]=warehouses_carriers["ontime_per"].apply(lambda x:pd.Series(classify(x)))

warehouses_carriers.sort_values(ascending=True,by="ontime_per")


,Warehouse_ID,Carrier,total_orders,order_value,shippingcost,ontime_orders,delayed_orders,ontime_per,delayed_per,class,Performance
20,WH-005,DHL,6642,7488.673528,133.926939,1649,4993,24.83,75.17,ClassC,Critical
5,WH-002,DHL,6485,7511.810244,136.953062,1634,4851,25.20,74.80,ClassC,Critical
35,WH-008,DHL,6586,7534.577083,137.441297,1686,4900,25.60,74.40,ClassC,Critical
39,WH-008,USPS,8621,7495.187180,136.623843,2673,5948,31.01,68.99,ClassC,Critical
9,WH-002,USPS,8651,7534.979060,136.934402,2709,5942,31.31,68.69,ClassC,Critical
24,WH-005,USPS,8794,7461.065650,137.268129,2903,5891,33.01,66.99,ClassC,Critical
7,WH-002,OnTrac,4357,7499.782447,137.947165,1527,2830,35.05,64.95,ClassC,Critical
37,WH-008,OnTrac,4265,7438.714851,137.126014,1511,2754,35.43,64.57,ClassC,Critical
38,WH-008,UPS,11865,7459.944592,136.350013,4318,7547,36.39,63.61,ClassC,Critical
22,WH-005,OnTrac,4369,7489.702456,137.059169,1610,2759,36.85,63.15,ClassC,Critical


In [87]:
warehouses_carriers_products=raw_data.groupby(["Warehouse_ID","Carrier","Product_Category"]).agg(
    total_orders=("Shipment_ID","count"),
    order_value=("Order_Value","mean"),
    shippingcost=("Shipping_Cost","mean"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())
).reset_index()
warehouses_carriers_products["ontime_per"]=round(warehouses_carriers_products['ontime_orders']/warehouses_carriers_products['total_orders']*100,2)
warehouses_carriers_products["delayed_per"]=round(warehouses_carriers_products['delayed_orders']/warehouses_carriers_products['total_orders']*100,2)

warehouses_carriers_products.sort_values(ascending=True,by="ontime_per")

def classify(ontime_per):
  if ontime_per<40:
    return "ClassC","Critical"

  elif ontime_per>=40 and ontime_per<=50:
    return "ClassB","Watch"

  else:
    return "ClassA","Excellent"
warehouses_carriers_products[["class","Performance"]]=warehouses_carriers_products["ontime_per"].apply(lambda x:pd.Series(classify(x)))

warehouses_carriers_products.sort_values(ascending=True,by="ontime_per").head(60)

,Warehouse_ID,Carrier,Product_Category,total_orders,order_value,shippingcost,ontime_orders,delayed_orders,ontime_per,delayed_per,class,Performance
47,WH-002,DHL,Pharmaceuticals,779,7444.251489,140.475469,151,628,19.38,80.62,ClassC,Critical
286,WH-008,DHL,Medical Supplies,516,7272.275233,136.130174,111,405,21.51,78.49,ClassC,Critical
167,WH-005,DHL,Pharmaceuticals,812,7491.534052,129.865899,181,631,22.29,77.71,ClassC,Critical
162,WH-005,DHL,Electronics,1277,7282.317400,126.618504,289,988,22.63,77.37,ClassC,Critical
42,WH-002,DHL,Electronics,1253,7457.516433,133.061341,286,967,22.83,77.17,ClassC,Critical
45,WH-002,DHL,Industrial,642,7739.398302,140.487056,147,495,22.90,77.10,ClassC,Critical
166,WH-005,DHL,Medical Supplies,532,7075.447914,139.295526,122,410,22.93,77.07,ClassC,Critical
282,WH-008,DHL,Electronics,1332,7616.362267,140.741149,324,1008,24.32,75.68,ClassC,Critical
287,WH-008,DHL,Pharmaceuticals,780,7477.980013,143.523051,194,586,24.87,75.13,ClassC,Critical
281,WH-008,DHL,Automotive,642,7526.459626,132.365358,160,482,24.92,75.08,ClassC,Critical


In [92]:
products=raw_data.groupby(["Product_Category"]).agg(
    total_orders=("Shipment_ID","count"),
    wight=("Product_Weight_kg","mean"),
    order_value=("Order_Value","mean"),
    shippingcost=("Shipping_Cost","mean"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())
).reset_index()
products["ontime_per"]=round(products['ontime_orders']/products['total_orders']*100,2)
products["delayed_per"]=round(products['delayed_orders']/products['total_orders']*100,2)

products.sort_values(ascending=True,by="ontime_per")

,Product_Category,total_orders,wight,order_value,shippingcost,ontime_orders,delayed_orders,ontime_per,delayed_per
7,Pharmaceuticals,52543,249.715146,7485.304008,138.247563,24124,28419,45.91,54.09
2,Electronics,87154,249.469399,7532.942563,137.015484,40025,47129,45.92,54.08
6,Medical Supplies,34927,250.547417,7531.088463,137.837625,16086,18841,46.06,53.94
0,Apparel,52279,249.696665,7481.292657,138.229294,26448,25831,50.59,49.41
5,Industrial,43804,249.683700,7505.774175,137.866055,22221,21583,50.73,49.27
3,Food & Beverage,56741,250.262055,7469.658956,138.993748,28825,27916,50.80,49.20
4,Furniture,65138,250.074401,7518.144855,138.522222,33201,31937,50.97,49.03
1,Automotive,43742,249.232426,7508.999604,137.672683,22300,21442,50.98,49.02


In [94]:
warehouses_carriers_Route=raw_data.groupby(["Warehouse_ID","Carrier","Route_ID"]).agg(
    total_orders=("Shipment_ID","count"),
    order_value=("Order_Value","mean"),
    shippingcost=("Shipping_Cost","mean"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())
).reset_index()
warehouses_carriers_Route["ontime_per"]=round(warehouses_carriers_Route['ontime_orders']/warehouses_carriers_Route['total_orders']*100,2)
warehouses_carriers_Route["delayed_per"]=round(warehouses_carriers_Route['delayed_orders']/warehouses_carriers_Route['total_orders']*100,2)

warehouses_carriers_Route.sort_values(ascending=True,by="ontime_per")

def classify(ontime_per):
  if ontime_per<40:
    return "ClassC","Critical"

  elif ontime_per>=40 and ontime_per<=50:
    return "ClassB","Watch"

  else:
    return "ClassA","Excellent"
warehouses_carriers_Route[["class","Performance"]]=warehouses_carriers_Route["ontime_per"].apply(lambda x:pd.Series(classify(x)))

warehouses_carriers_Route.sort_values(ascending=True,by="ontime_per").head(50)


,Warehouse_ID,Carrier,Route_ID,total_orders,order_value,shippingcost,ontime_orders,delayed_orders,ontime_per,delayed_per,class,Performance
4188,WH-005,DHL,RT-0189,35,6643.704571,164.302000,1,34,2.86,97.14,ClassC,Critical
1172,WH-002,DHL,RT-0173,30,6221.643333,155.744333,1,29,3.33,96.67,ClassC,Critical
173,WH-001,DHL,RT-0174,21,5580.190952,110.190476,1,20,4.76,95.24,ClassC,Critical
1508,WH-002,OnTrac,RT-0109,21,8621.900000,98.000000,1,20,4.76,95.24,ClassC,Critical
7017,WH-008,DHL,RT-0018,35,6179.599143,156.532000,2,33,5.71,94.29,ClassC,Critical
1588,WH-002,OnTrac,RT-0189,17,6059.067647,125.254706,1,16,5.88,94.12,ClassC,Critical
1097,WH-002,DHL,RT-0098,32,6759.061563,109.223750,2,30,6.25,93.75,ClassC,Critical
1023,WH-002,DHL,RT-0024,31,7370.389032,155.566452,2,29,6.45,93.55,ClassC,Critical
1188,WH-002,DHL,RT-0189,31,6818.100968,120.960000,2,29,6.45,93.55,ClassC,Critical
7417,WH-008,OnTrac,RT-0018,15,7905.384000,117.429333,1,14,6.67,93.33,ClassC,Critical


In [98]:
customer=raw_data.groupby(["Customer_Segment"]).agg(
    total_orders=("Shipment_ID","count"),
    order_value=("Order_Value","mean"),
    shippingcost=("Shipping_Cost","mean"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())
).reset_index()
customer["ontime_per"]=round(customer['ontime_orders']/customer['total_orders']*100,2)
customer["delayed_per"]=round(customer['delayed_orders']/customer['total_orders']*100,2)

customer.sort_values(ascending=True,by="ontime_per")

,Customer_Segment,total_orders,order_value,shippingcost,ontime_orders,delayed_orders,ontime_per,delayed_per
2,Government,43556,7512.465399,138.134012,21226,22330,48.73,51.27
1,B2C,195883,7497.999221,137.588749,95715,100168,48.86,51.14
0,B2B,196889,7510.984041,138.398484,96289,100600,48.91,51.09


Monthly Analysis

In [103]:
raw_data["Order_Date"]=pd.to_datetime(raw_data["Order_Date"])
raw_data["Promised_Delivery_Date"]=pd.to_datetime(raw_data["Promised_Delivery_Date"])
raw_data["Actual_Delivery_Date"]=pd.to_datetime(raw_data["Actual_Delivery_Date"])

In [133]:
avg_delay=raw_data['Actual_Delivery_Date']-raw_data['Promised_Delivery_Date']
raw_data["avg_delay"]=avg_delay.dt.days


In [134]:
warehouses_carriers_Route=raw_data.groupby(["Warehouse_ID","Carrier"]).agg(
    total_orders=("Shipment_ID","count"),
    avg_delay=("avg_delay","mean"),
    max_delay=("avg_delay","max"),
    min_delay=("avg_delay","min"),
    order_value=("Order_Value","mean"),
    shippingcost=("Shipping_Cost","mean"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())
).reset_index()
warehouses_carriers_Route["ontime_per"]=round(warehouses_carriers_Route['ontime_orders']/warehouses_carriers_Route['total_orders']*100,2)
warehouses_carriers_Route["delayed_per"]=round(warehouses_carriers_Route['delayed_orders']/warehouses_carriers_Route['total_orders']*100,2)

warehouses_carriers_Route.sort_values(ascending=True,by="ontime_per")

def classify(ontime_per):
  if ontime_per<40:
    return "ClassC","Critical"

  elif ontime_per>=40 and ontime_per<=50:
    return "ClassB","Watch"

  else:
    return "ClassA","Excellent"
warehouses_carriers_Route[["class","Performance"]]=warehouses_carriers_Route["ontime_per"].apply(lambda x:pd.Series(classify(x)))

warehouses_carriers_Route.sort_values(ascending=True,by="ontime_per").head(50)


,Warehouse_ID,Carrier,total_orders,avg_delay,max_delay,min_delay,order_value,shippingcost,ontime_orders,delayed_orders,ontime_per,delayed_per,class,Performance
20,WH-005,DHL,6642,6.598916,62,-3,7488.673528,133.926939,1649,4993,24.83,75.17,ClassC,Critical
5,WH-002,DHL,6485,6.541403,60,-3,7511.810244,136.953062,1634,4851,25.20,74.80,ClassC,Critical
35,WH-008,DHL,6586,6.516095,61,-3,7534.577083,137.441297,1686,4900,25.60,74.40,ClassC,Critical
39,WH-008,USPS,8621,6.044078,61,-3,7495.187180,136.623843,2673,5948,31.01,68.99,ClassC,Critical
9,WH-002,USPS,8651,6.002081,58,-3,7534.979060,136.934402,2709,5942,31.31,68.69,ClassC,Critical
24,WH-005,USPS,8794,5.799636,63,-3,7461.065650,137.268129,2903,5891,33.01,66.99,ClassC,Critical
7,WH-002,OnTrac,4357,5.476475,61,-3,7499.782447,137.947165,1527,2830,35.05,64.95,ClassC,Critical
37,WH-008,OnTrac,4265,5.549824,58,-3,7438.714851,137.126014,1511,2754,35.43,64.57,ClassC,Critical
38,WH-008,UPS,11865,5.311083,60,-3,7459.944592,136.350013,4318,7547,36.39,63.61,ClassC,Critical
22,WH-005,OnTrac,4369,5.390936,60,-3,7489.702456,137.059169,1610,2759,36.85,63.15,ClassC,Critical


In [135]:
raw_data["year"]=raw_data['Order_Date'].dt.year
raw_data["month"]=raw_data['Order_Date'].dt.month
raw_data["month_name"]=raw_data['Order_Date'].dt.strftime("%b")
raw_data["quater"]=raw_data['Order_Date'].dt.quarter


raw_data["day"]=raw_data['Order_Date'].dt.day
raw_data["day_name"]=raw_data['Order_Date'].dt.strftime("%a")


In [136]:
monthly_ontime=raw_data.groupby(["year","month","month_name"]).agg(
    total_orders=("Shipment_ID","count"),
    ontime_orders=("On_Time_Flag",lambda x:(x==1).sum()),
    delayed_orders=("On_Time_Flag",lambda x:(x==0).sum())).reset_index()
monthly_ontime["ontime_per"]=round(monthly_ontime['ontime_orders']/monthly_ontime['total_orders']*100,2)
monthly_ontime["delayed_per"]=round(monthly_ontime['delayed_orders']/monthly_ontime['total_orders']*100,2)
monthly_ontime.loc[monthly_ontime["ontime_per"]<=40]

,year,month,month_name,total_orders,ontime_orders,delayed_orders,ontime_per,delayed_per
10,2021,11,Nov,9046,2980,6066,32.94,67.06
11,2021,12,Dec,9221,2990,6231,32.43,67.57
22,2022,11,Nov,9069,2944,6125,32.46,67.54
23,2022,12,Dec,9287,3123,6164,33.63,66.37
34,2023,11,Nov,8996,2932,6064,32.59,67.41
35,2023,12,Dec,9313,3037,6276,32.61,67.39
46,2024,11,Nov,8944,2931,6013,32.77,67.23
47,2024,12,Dec,9332,3089,6243,33.10,66.90


In [141]:
quarterly = raw_data.groupby('quater').agg(
    Total=('Shipment_ID','count'),
    OnTime_Rate=('On_Time_Flag', lambda x: x.mean()*100),
    delayed_per=('On_Time_Flag', lambda x: 100 - x.mean()*100)
).reset_index()
print(quarterly)

   quater   Total  OnTime_Rate  delayed_per
0       1  107950    47.641501    52.358499
1       2  108436    54.389686    45.610314
2       3  109778    53.049791    46.950209
3       4  110164    40.472387    59.527613


In [137]:
delayed = raw_data[raw_data['On_Time_Flag']==0]
print(f"Total order value at risk: ${delayed['Order_Value'].sum():,.2f}")
print(f"Total shipping cost wasted: ${delayed['Shipping_Cost'].sum():,.2f}")

Total order value at risk: $1,676,069,083.73
Total shipping cost wasted: $30,763,303.55


Final check on nulls or dupicates or any changes

In [142]:
print(f" nulls:{raw_data.isnull().sum()},dupicates:{raw_data.duplicated().sum()},shape:{raw_data.shape}")
print(f"{raw_data.info()} ,{raw_data.describe()}")

 nulls:Shipment_ID               0
Order_Date                0
Promised_Delivery_Date    0
Actual_Delivery_Date      0
Delay_Days                0
Delay_Status              0
On_Time_Flag              0
Origin_City               0
Origin_State              0
Origin_Region             0
Destination_City          0
Destination_State         0
Destination_Region        0
Route_ID                  0
Distance_Miles            0
Warehouse_ID              0
Warehouse_City            0
Warehouse_State           0
Carrier                   0
Ship_Mode                 0
Product_Category          0
Product_Weight_kg         0
Order_Value               0
Shipping_Cost             0
Customer_Segment          0
Customer_ID               0
Weather_Condition         0
Traffic_Condition         0
avg_delay                 0
year                      0
month                     0
day                       0
quater                    0
month_name                0
day_name                  0
dtype: int64,

In [143]:
raw_data.to_csv("Cleaned_data_supply_chain.csv",index=False)